# 02 - Data Cleaning & Feature Engineering (Production Side)

**Stage 3 of the project.**

This notebook takes the raw production-side tables (`fact_production_plan_raw`,
`fact_production_raw`, `fact_downtime_raw`, `fact_material_consumption_raw`)
and turns them into the `datasets/processed/*_processed.csv` files that get
loaded into MySQL in the next notebook.

All the transformation *logic* lives in `etl_lib.py`, sitting next to this
notebook. Every function there has a docstring explaining not just *what*
it does but *why* -- read it alongside this notebook if you want the full
picture.

### Why Python (not SQL) for this stage?
The cleaning rules are **row-by-row and stateful** (e.g. the raw-material
lot sequence number in `LotId` depends on comparing each row to the
*previous* row for the same work order) -- this is awkward to express in
a single SQL statement, but a natural `for` loop in pandas. MySQL only
enters the picture *after* the data is clean -- it is the serving layer
for BI tools, not the place we want to debug data-quality logic.


In [1]:
import sys
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import etl_lib as etl

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

RAW = '../../datasets/raw'
DIM = '../../datasets/dim'
PROCESSED = '../../datasets/processed'
SHIFT_MAP = {'Shift 1': 1, 'Shift 2': 2, 'Shift 3': 3}


In [2]:
plan = pd.read_csv(f'{RAW}/fact_production_plan_raw.csv', encoding='utf-8-sig')
prod = pd.read_csv(f'{RAW}/fact_production_raw.csv', encoding='utf-8-sig')
downtime = pd.read_csv(f'{RAW}/fact_downtime_raw.csv', encoding='utf-8-sig')
matcons = pd.read_csv(f'{RAW}/fact_material_consumption_raw.csv', encoding='utf-8-sig')
machine_setup = pd.read_csv(f'{DIM}/dim_machine_setup.csv', encoding='utf-8-sig')

print({k: v.shape for k, v in {'plan': plan, 'prod': prod, 'downtime': downtime, 'matcons': matcons}.items()})


{'plan': (9128, 10), 'prod': (9137, 13), 'downtime': (21874, 10), 'matcons': (23504, 10)}


## 3.1 Generic cleaning

For every fact table we apply, in this order:

1. **`normalize_placeholder_nulls`** -- turn `-`, `--`, `/`, `//`, a lone
   space, etc. into a real `NaN`.
2. **`strip_whitespace`** -- collapse internal double-spaces and trim
   leading/trailing spaces from every text column.
3. **`standardize_categories`** -- fold `INJECTION MOLDING` / `injection
   molding` / `Injection Molding ` into one canonical `Injection Molding`.
4. **`fix_negative_quantities`** -- `abs()` on quantity columns that
   should never be negative (`RejectedQty`, `EndWeightKg`).
5. **`drop_exact_duplicates`** -- remove fully identical rows. We exclude
   any synthetic row-id column (`RecordSeq`) from the duplicate check,
   otherwise two genuinely identical business records with different
   row-ids would never be detected as duplicates.

We print how many duplicate rows were removed from each table -- silently
dropping rows without reporting it is exactly what makes a cleaning
pipeline hard to trust.


In [3]:
def clean_generic(df, cat_cols, neg_cols, label):
    df = etl.normalize_placeholder_nulls(df)
    df = etl.strip_whitespace(df)
    df = etl.standardize_categories(df, cat_cols)
    df = etl.fix_negative_quantities(df, neg_cols)
    id_cols = [c for c in df.columns if c != 'RecordSeq']
    df, n_dupes = etl.drop_exact_duplicates(df, subset=id_cols)
    print(f"[{label}] removed {n_dupes} duplicate rows -> {len(df):,} rows remain")
    return df

plan = clean_generic(plan, ['Process'], [], 'plan')
prod = clean_generic(prod, ['Process'], ['RejectedQty'], 'production')
downtime = clean_generic(downtime, ['Process', 'Shift'], [], 'downtime')
matcons = clean_generic(matcons, ['Process', 'Shift'], ['EndWeightKg'], 'material_consumption')


[plan] removed 18 duplicate rows -> 9,110 rows remain


[production] removed 27 duplicate rows -> 9,110 rows remain


[downtime] removed 65 duplicate rows -> 21,809 rows remain


[material_consumption] removed 70 duplicate rows -> 23,434 rows remain


In [4]:
print("Process categories after cleaning:", sorted(prod['Process'].unique()))
print("Negative RejectedQty remaining:", (prod['RejectedQty'] < 0).sum())
print("Negative EndWeightKg remaining:", (matcons['EndWeightKg'] < 0).sum())


Process categories after cleaning: ['Blow Molding', 'Hot Foil Stamping', 'Injection Molding', 'Screen Printing']
Negative RejectedQty remaining: 0
Negative EndWeightKg remaining: 0


### A domain-specific cleaning rule: forward-filling missing material lots

A missing `MaterialLot` (cleaned from a `-` placeholder) almost certainly
means the operator forgot to log which lot was in use, **not** that a new
physical lot suddenly appeared. If left as `NaN`, the lot-change sequence
logic below would misread every missing value as "a new lot started
here", inflating the change counter for no real reason.


In [5]:
matcons['MaterialLot'] = etl.ffill_material_lot(matcons)
matcons['MaterialLot'].isna().sum()  # should be 0 unless an entire work order has no material data at all


np.int64(0)

## 3.2 Calendar columns: `ISOWeek` and `ISOWeekday`

Manufacturing overtime and shift-pattern rules are almost always expressed
in **ISO calendar weeks** (Monday-Sunday) rather than plain calendar
weeks, because it gives every week a consistent 7-day span. `ISOWeek` is
the ISO week number, `ISOWeekday` is the ISO weekday (1=Monday ...
7=Sunday).


In [6]:
for df in (plan, prod, downtime, matcons):
    cal = etl.add_iso_calendar_columns(df, 'Date')
    df[['ISOWeek', 'ISOWeekday']] = cal[['ISOWeek', 'ISOWeekday']]

prod[['Date', 'ISOWeek', 'ISOWeekday']].head()


,Date,ISOWeek,ISOWeekday
0,2025-12-04,49,4
1,2026-05-19,21,2
2,2025-08-12,33,2
3,2025-07-30,31,3
4,2025-12-01,49,1


## 3.3 `ShiftNumber`: numeric shift indicator

Reproduces the Excel formula from the project brief:

```
=IF(AND(StartTime>=06:00, StartTime<14:00), 1,
    IF(AND(StartTime>=14:00, StartTime<22:00), 2, 3))
```

`fact_production_raw` and `fact_production_plan_raw` only have a
`StartTime` column, so `ShiftNumber` is computed from it directly.
`fact_downtime_raw` and `fact_material_consumption_raw` already carry a
`Shift` text column from the MES, which we simply map to its numeric
equivalent.


In [7]:
plan['ShiftNumber'] = etl.compute_shift_number(plan['StartTime'])
prod['ShiftNumber'] = etl.compute_shift_number(prod['StartTime'])
downtime['ShiftNumber'] = downtime['Shift'].map(SHIFT_MAP)
matcons['ShiftNumber'] = matcons['Shift'].map(SHIFT_MAP)

prod[['StartTime', 'ShiftNumber']].sample(8, random_state=1)


,StartTime,ShiftNumber
6458,05:46:00,3
976,18:33:00,2
834,09:17:00,1
6169,22:40:00,3
956,11:51:00,1
2012,19:39:00,2
7043,01:44:00,3
2984,12:41:00,1


## 3.4 `LotId`: the batch traceability code

This is the most involved rule in the brief, so it's worth spelling out
carefully. `LotId` is a 16-character string, built by concatenating:

| Segment | Digits | Example | Meaning |
|---|---|---|---|
| Year | 2 | `26` | last 2 digits of the calendar year |
| ISO week | 2 | `09` | `ISOWeek` |
| ISO weekday | 1 | `2` | `ISOWeekday` (1=Mon...7=Sun) |
| Shift | 1 | `2` | `ShiftNumber` |
| Process | 1 | `1` | Blow Molding=1, Injection Molding=2, Screen Printing=4, Hot Foil Stamping=5 |
| Machine number | 2 | `01` | numeric suffix of `MachineId` (`ISBM-001` -> `01`) |
| Work order number | 5 | `06005` | numeric part of the work order `WO-6005` |
| Material lot sequence | 2 | `01` | see below |

**The last 2 digits are the interesting part.** They start at `01` for
every new work order and increase by 1 every time the *physical*
raw-material lot being consumed on that order changes -- tracked in
`fact_material_consumption_raw`. A shift change alone (Shift 2 -> Shift 3)
does **not** bump this counter if the same material lot is still being
used; only an actual lot substitution does.

This is exactly why `fact_material_consumption_raw` gets **two** lot
columns, `LotIdStart` and `LotIdEnd`: the first 14 characters (year,
week, weekday, *shift*) can differ between the start and the end of a
single consumption record if it happens to straddle a shift boundary,
while the last 2 digits (the material sequence) stay identical -- because
it is still the same physical lot.


In [8]:
# Step 1: the material-lot change sequence, computed from fact_material_consumption_raw
# (needs a stable chronological order -- RecordSeq is the MES's own auto-increment
# record id, which survives even though the CSV export itself may be re-sorted)
matcons['MaterialLotSeq'] = etl.assign_material_lot_sequence(matcons)

matcons.sort_values(['WorkOrder', 'RecordSeq'])[['WorkOrder', 'RecordSeq', 'MaterialLot', 'MaterialLotSeq']].head(10)


,WorkOrder,RecordSeq,MaterialLot,MaterialLotSeq
10234,WO-10000,18496,COL-COR-004-2432,1
5890,WO-10000,18497,COL-COR-004-2433,2
17461,WO-10000,18579,COL-COR-004-2435,3
9420,WO-10000,18583,COL-COR-004-2434,4
8627,WO-10001,18580,COL-COR-004-2436,1
7093,WO-10001,18581,COL-COR-004-2437,2
16195,WO-10001,18582,COL-COR-004-2438,3
12052,WO-10001,18598,COL-COR-004-2439,4
270,WO-10002,18666,COL-COR-007-2778,1
4998,WO-10002,18667,COL-COR-007-2779,2


In [9]:
# Step 2: the 14-character prefix, then append the 2-digit material sequence
matcons['LotIdPrefix'] = etl.build_lot_prefix(
    matcons['Date'], matcons['ShiftNumber'], matcons['Process'], matcons['MachineId'], matcons['WorkOrder'])
matcons['LotIdStart'] = matcons['LotIdPrefix'] + matcons['MaterialLotSeq'].astype(str).str.zfill(2)
# a single consumption record represents one continuous stretch of the same physical
# lot, so LotIdStart and LotIdEnd coincide for that record; a genuinely different
# lot always becomes a *separate* row via assign_material_lot_sequence above.
matcons['LotIdEnd'] = matcons['LotIdStart']
matcons = matcons.drop(columns=['LotIdPrefix'])

assert matcons['LotIdStart'].str.len().nunique() == 1, "LotId must always be 16 characters"
matcons[['WorkOrder', 'MaterialLotSeq', 'LotIdStart', 'LotIdEnd']].head()


,WorkOrder,MaterialLotSeq,LotIdStart,LotIdEnd
0,WO-1314,1,2610115010131401,2610115010131401
1,WO-3119,2,2540232030311902,2540232030311902
2,WO-9360,4,2605234010936004,2605234010936004
3,WO-6239,2,2545311030623902,2545311030623902
4,WO-7554,4,2625331050755404,2625331050755404


### `LotId` on `fact_production_raw`

A work order is a single row in `fact_production_raw`, but it can consume
more than one material lot over its run. We link production to its
**first** material lot (the one active at `StartTime`) -- this is a
deliberate simplification: if you need the *full* lot history of an
order, join out to `fact_material_consumption_raw` on `WorkOrder`, which
retains every lot change individually.


In [10]:
prod['LotIdPrefix'] = etl.build_lot_prefix(prod['Date'], prod['ShiftNumber'], prod['Process'], prod['MachineId'], prod['WorkOrder'])

first_seq = matcons.sort_values(['WorkOrder', 'RecordSeq']).groupby('WorkOrder')['MaterialLotSeq'].first()
prod['MaterialLotSeq'] = prod['WorkOrder'].map(first_seq).fillna(1).astype(int)
prod['LotId'] = prod['LotIdPrefix'] + prod['MaterialLotSeq'].astype(str).str.zfill(2)
prod = prod.drop(columns=['LotIdPrefix'])

prod[['Date', 'ShiftNumber', 'Process', 'MachineId', 'WorkOrder', 'LotId']].head()


,Date,ShiftNumber,Process,MachineId,WorkOrder,LotId
0,2025-12-04,1,Hot Foil Stamping,HF-002,WO-1703,2549415020170301
1,2026-05-19,2,Blow Molding,ISBM-007,WO-8533,2621221070853301
2,2025-08-12,3,Blow Molding,ISBM-003,WO-6123,2533231030612301
3,2025-07-30,3,Blow Molding,ISBM-002,WO-5594,2531331020559401
4,2025-12-01,1,Injection Molding,IM-002,WO-2703,2549112020270301


### `LotId` on `fact_downtime_raw`

Downtime events don't carry a `WorkOrder` column of their own -- a machine
stoppage isn't "produced" by an order the way a batch is. To trace a
stoppage back to the order that was running when it happened, we resolve
each work order's true start/end **timestamp** (not just a time-of-day,
since an order can run past midnight) and match each downtime event's
timestamp into that interval, per machine.


In [11]:
# resolve absolute start/end datetimes for every work order
prod = prod.merge(plan[['WorkOrder', 'PlannedHours']], on='WorkOrder', how='left')
prod['_start_dt'] = pd.to_datetime(prod['Date']) + pd.to_timedelta(prod['StartTime'].astype(str))
prod['_end_dt'] = prod['_start_dt'] + pd.to_timedelta(prod['PlannedHours'].astype(float), unit='h')

downtime['MatchedWorkOrder'] = etl.match_downtime_to_work_order(downtime, prod)
wo_to_lot = prod.set_index('WorkOrder')['LotId']
downtime['LotId'] = downtime['MatchedWorkOrder'].map(wo_to_lot)

match_rate = downtime['MatchedWorkOrder'].notna().mean()
print(f"downtime events successfully matched to a work order: {match_rate:.1%}")
print("(the remainder are stoppages logged during a changeover/idle gap between two orders, "
      "or right at the very start/end of the dataset's time horizon)")
downtime[['Date', 'MachineId', 'StoppageStartTime', 'MatchedWorkOrder', 'LotId']].dropna().head()


downtime events successfully matched to a work order: 84.8%
(the remainder are stoppages logged during a changeover/idle gap between two orders, or right at the very start/end of the dataset's time horizon)


,Date,MachineId,StoppageStartTime,MatchedWorkOrder,LotId
0,2026-03-02,IM-003,22:11:00,WO-3327,2610112030332701
1,2025-07-25,SS-001,10:56:00,WO-9114,2530424010911401
2,2026-01-12,IM-003,07:10:00,WO-3259,2603112030325901
3,2025-07-21,ISBM-007,20:44:00,WO-8112,2530121070811201
4,2025-11-21,IM-005,04:23:00,WO-4212,2547532050421201


## 3.5 Maintenance flags on `fact_downtime_raw`

Adds the duration in decimal minutes (handling the overnight wrap) and
three boolean flags used throughout the maintenance KPIs: whether the
event is a genuine unplanned failure (used for MTBF/MTTR -- explicitly
excluding planned changeovers/cleaning/preventive maintenance, which are
not equipment failures), whether it's a tooling changeover, and whether
it's scheduled preventive maintenance.

**This flag is also what Stage 4 uses to split the downtime Pareto
analysis into two separate charts (planned vs. unplanned).** Mixing
planned and unplanned stoppages into a single Pareto is misleading: a
scheduled shift-3 meal break or a routine mold change will always be
near the top by sheer frequency, crowding out the *genuine reliability
problems* (mechanical/electrical failures, shortages) that a
continuous-improvement team should actually prioritize. Planned time is a
capacity-planning conversation; unplanned time is a reliability
conversation -- they need separate analysis.


In [12]:
downtime = etl.add_maintenance_flags(downtime)
downtime['LeadTimeProdHours'] = downtime['DowntimeDurationMin'] / 60.0

downtime[['StoppageReason', 'PlannedStoppage', 'DowntimeDurationMin',
          'UnplannedFailure', 'IsChangeoverSetup', 'IsPreventiveMaintenance']].drop_duplicates('StoppageReason')


,StoppageReason,PlannedStoppage,DowntimeDurationMin,UnplannedFailure,IsChangeoverSetup,IsPreventiveMaintenance
0,Raw Material Shortage,No,16.0,True,False,False
1,Mechanical Failure (Roller/Squeegee),No,35.0,True,False,False
2,Operator Unavailable,No,21.0,True,False,False
3,Quality Adjustment (Rejects),No,17.0,False,False,False
5,Color Registration Adjustment,No,11.0,False,False,False
6,"Meal Break (Shift 3 - no relief crew, 10min sh...",Yes,80.0,False,False,False
7,Utilities Shortage (Compressed Air),No,24.0,True,False,False
8,Planned Preventive Maintenance,Yes,89.0,False,False,True
10,Hot Foil Ribbon Shortage,No,19.0,True,False,False
11,Electrical/Control Failure,No,44.0,True,False,False


## 3.6 OEE building blocks on `fact_production_raw`

- **Availability** = Run Time / Planned Time, where Run Time excludes
  unplanned downtime linked to that order.
- **Performance** = (units produced / Run Time) / rated capacity
  (pieces/hour) of that machine+mold combination.
- **Quality** = (units produced - units rejected) / units produced. Note
  this is the **100%-inspection** yield -- different from the AQL
  *sample-based* accept/reject decision in the quality-control tables.
- **OEE** = Availability x Performance x Quality.
- **Actual cycle time** = Run Time / units produced (seconds/unit).
- **Setup time** = planned changeover-type downtime linked to that order.
- **Throughput lead time** = wall-clock time from this order's start to
  the *next* order's start on the same machine.


In [13]:
import re as _re

def _parse_rated_capacity(s):
    return int(_re.sub(r'[^\d]', '', str(s).split('/')[0]))

capacity_lookup = {(r['MachineId'], r['MoldId']): _parse_rated_capacity(r['RatedCapacityPerHour'])
                    for _, r in machine_setup.iterrows()}
# Screen Printing / Hot Foil Stamping are not in dim_machine_setup (documented
# assumption from the production-simulation stage): reuse the fixed capacities
# assumed there (SS ~900-950 pc/h, HF ~600-650 pc/h) directly as pieces/hour.
for m, cap_h in {'SS-001': 950, 'SS-002': 900, 'HF-001': 650, 'HF-002': 600}.items():
    for tool in prod.loc[prod['MachineId'] == m, 'ToolId'].unique():
        capacity_lookup[(m, tool)] = cap_h

prod['LeadTimeProdHours'] = etl.add_lead_time_prod(prod, duration_hours_col='PlannedHours')

downtime_for_oee = downtime.dropna(subset=['MatchedWorkOrder']).rename(columns={'MatchedWorkOrder': 'WorkOrder'})
prod_for_oee = prod.drop(columns=['PlannedHours'])
prod = etl.compute_oee_components(prod_for_oee, plan, downtime_for_oee, capacity_lookup)

prod[['WorkOrder', 'Process', 'Availability', 'Performance', 'Quality', 'OEE',
      'ActualCycleTimeSec', 'SetupTimeHours', 'ThroughputLeadTimeHours']].head()


,WorkOrder,Process,Availability,Performance,Quality,OEE,ActualCycleTimeSec,SetupTimeHours,ThroughputLeadTimeHours
7788,WO-1001,Hot Foil Stamping,0.945158,1.048488,0.997882,0.988888,5.282332,0.0,12.466667
5100,WO-1002,Hot Foil Stamping,1.000000,0.976403,0.997934,0.974386,5.672313,0.0,19.066667
4478,WO-1003,Hot Foil Stamping,1.000000,0.969576,0.998127,0.967760,5.712252,0.0,40.466667
4567,WO-1004,Hot Foil Stamping,1.000000,1.002249,0.998205,1.000450,5.526032,0.0,5.516667
5067,WO-1005,Hot Foil Stamping,1.000000,0.996262,0.998520,0.994788,5.559243,0.0,14.616667


In [14]:
print("Average OEE components by process:")
prod.groupby('Process')[['Availability', 'Performance', 'Quality', 'OEE']].mean().round(3)


Average OEE components by process:


,Availability,Performance,Quality,OEE
Process,,,,
Blow Molding,0.947,1.019,0.970,0.931
Hot Foil Stamping,0.974,0.988,0.998,0.959
Injection Molding,0.947,1.020,0.968,0.928
Screen Printing,0.970,0.992,0.995,0.956


OEE in the 0.90-0.96 range with Availability as the main drag (consistent
with the maintenance-rule design from the production simulation stage:
Hot Foil Stamping/Screen Printing run 24/7 with no shift-3 relief crew, so
they absorb a fixed daily stoppage that Injection Molding/Blow Molding are
less exposed to) is a realistic result for a well-run but not "world
class" plant -- world-class OEE benchmarks sit around 0.85 as a *good*
result and 0.40-0.60 is typical for an unmanaged process.


## 3.7 Hiding columns that don't help the analysis

*"Information that doesn't help you, gets in your way."* A handful of
columns exist only as scaffolding for the calculations above and add
nothing once `LotId`/OEE are computed -- carrying them into
`datasets/processed/` (and from there into MySQL and Power BI) would just
be noise on every table browse and every `SELECT *`. We drop them here,
right before saving:

- `_start_dt`, `_end_dt`, `_next_start_dt` -- internal timestamp helpers,
  fully superseded by `LotId` and the OEE time columns.
- `RecordSeq` (material consumption only) -- the MES auto-increment id
  used solely to reconstruct chronological order for
  `assign_material_lot_sequence`; once `MaterialLotSeq`/`LotIdStart` exist,
  it has no analytical value.

Everything else stays, including intermediate-looking columns like
`MaterialLotSeq` (kept because it's a human-readable complement to the
encoded `LotId`, not because it's needed for any further calculation).


In [15]:
drop_helper_cols = ['_start_dt', '_end_dt', '_next_start_dt']
plan.to_csv(f'{PROCESSED}/fact_production_plan_processed.csv', encoding='utf-8-sig', index=False)
prod.drop(columns=[c for c in drop_helper_cols if c in prod.columns]).to_csv(
    f'{PROCESSED}/fact_production_processed.csv', encoding='utf-8-sig', index=False)
downtime.to_csv(f'{PROCESSED}/fact_downtime_processed.csv', encoding='utf-8-sig', index=False)
matcons.drop(columns=['RecordSeq']).to_csv(f'{PROCESSED}/fact_material_consumption_processed.csv', encoding='utf-8-sig', index=False)

print("Processed production-side files written to datasets/processed/:")
for name, df in {'plan': plan, 'production': prod, 'downtime': downtime, 'material_consumption': matcons}.items():
    print(f"  {name}: {df.shape}")


Processed production-side files written to datasets/processed/:
  plan: (9110, 13)
  production: (9110, 35)
  downtime: (21809, 20)
  material_consumption: (23434, 16)
